# Session 4 - segment EDA, leakage-safe features, first global LightGBM

**Revised after external review caught two real methodology problems in the first
pass** (see DECISIONS.md D007/D008 for the full account): the internal train/validation
split had overlapping target periods (unpurged), and the objective (Tweedie vs L1 vs L2)
was selected by comparing scores on the true 13-week holdout -- which quietly turns the
holdout into a model-selection set. Both are fixed here.

1. Targeted EDA: active / intermittent / sparse segments (analytical thresholds, not causal categories).
2. Leakage-safe direct multi-horizon feature matrix (DECISIONS.md D006), with a **purged**
   internal train/validation split (D008).
3. Select the LightGBM objective using the internal validation set **only**.
4. Take exactly one look at the real 13-week holdout with the selected model.
5. Compare against the full Session 3 baseline suite.
6. Score components, signed bias, non-negative predictions, and a properly-quantified
   by-segment breakdown (demand share / error share / bias, not just a bare score).

In [1]:
import os
from pathlib import Path

current = Path.cwd()
while not (current / "CONTEXT.md").is_file():
    if current == current.parent:
        raise FileNotFoundError("Could not find CONTEXT.md in any parent directory")
    current = current.parent
os.chdir(current)
print("repo root:", Path.cwd())

import numpy as np
import pandas as pd
import lightgbm as lgb
print("lightgbm version:", lgb.__version__)

from src.data import KEY
from src.features import (
    build_direct_horizon_matrix, purge_fit_origins, zero_rate_table, classify_segment,
)
from src.backtest import (
    split_last_horizon, ma12_forecast, zero_forecast, seasonal_naive_forecast,
    score_forecast, HORIZON,
)
from src.metrics import vn1_score, vn1_score_breakdown

df = pd.read_parquet("data/processed/vn1_long.parquet")
train, test = split_last_horizon(df, horizon=HORIZON)
print("train weeks:", train["date"].nunique(), " test weeks:", test["date"].nunique())

repo root: D:\vn1-forecasting


lightgbm version: 4.7.0


train weeks: 157  test weeks: 13


## 1. Segment EDA (train-only, so nothing here can see the holdout)

Analytical thresholds on the observed zero-rate - a convenience for grouping series
with similar behavior, not a causal claim about why a series behaves that way (same
positional-not-causal discipline as Session 3's zero-position labels):

- **active** - zero_rate < 0.3
- **intermittent** - 0.3 <= zero_rate < 0.9
- **sparse** - zero_rate >= 0.9

In [2]:
# Cell - segment counts and behavior, computed from TRAIN only
zr = zero_rate_table(train)
zr["segment"] = classify_segment(zr["zero_rate"], active_max=0.3, sparse_min=0.9)

print("segment counts:")
print(zr["segment"].value_counts())
print()
print("mean_sales / zero_rate by segment:")
print(zr.groupby("segment", observed=True)[["mean_sales", "zero_rate"]].mean().round(3))

segment counts:
segment
sparse          6597
intermittent    6450
active          2006
Name: count, dtype: int64

mean_sales / zero_rate by segment:
              mean_sales  zero_rate
segment                            
active            65.720      0.117
intermittent       7.883      0.675
sparse             0.259      0.972


## 2. Leakage-safe direct multi-horizon feature matrix, with a purged validation split

Per DECISIONS.md D006: one row per (series, origin, horizon); every lag/rolling/price
feature is computed using only data at or before the origin, `horizon` (1-13) is an
explicit feature, and only `target`/calendar-of-target-date may reference a date after
the origin (calendar structure is always knowable in advance -- that's not leakage).

**Purged split (DECISIONS.md D008):** naively taking "the last few origins" as an internal
validation set isn't enough on its own -- a fit origin whose horizon reaches into or past
the first validation origin trains the model on a target from a calendar date the
validation set is nominally checking generalization to. `purge_fit_origins()` drops any
such fit origin, and `max(fit_origins) + max(horizons) < min(val_origins)` is asserted
explicitly rather than assumed. The one evaluation origin (156 = the last training week)
predicts exactly the 13 real test dates -- verified to line up with `split_last_horizon`'s
own test set, and is never part of fit or validation.

In [3]:
# Cell - build fit/validation/eval feature matrices, with a purged fit/val boundary
origins = list(range(52, 144, 4))
horizons = list(range(1, 14))
val_origins = origins[-3:]
fit_origins = purge_fit_origins(origins, val_origins, horizons)
assert max(fit_origins) + max(horizons) < min(val_origins)
print("fit_origins (purged):", fit_origins)
print("val_origins:", val_origins)

train_mat = build_direct_horizon_matrix(df, origins=fit_origins, horizons=horizons)
val_mat = build_direct_horizon_matrix(df, origins=val_origins, horizons=horizons)
eval_mat = build_direct_horizon_matrix(df, origins=[156], horizons=horizons)

feature_cols = [
    c for c in train_mat.columns if c not in (*KEY, "origin_date", "target_date", "target")
]
for m in (train_mat, val_mat, eval_mat):
    m["segment"] = m["segment"].astype("category")

print("fit rows:", len(train_mat), " val rows:", len(val_mat), " eval rows:", len(eval_mat))
print("eval target dates match test dates:",
      sorted(eval_mat["target_date"].unique()) == sorted(test["date"].unique()))
print("n_features:", len(feature_cols))
feature_cols

fit_origins (purged): [52, 56, 60, 64, 68, 72, 76, 80, 84, 88, 92, 96, 100, 104, 108, 112, 116]
val_origins: [132, 136, 140]


fit rows: 3326713  val rows: 587067  eval rows: 195689
eval target dates match test dates: True
n_features: 22


['horizon',
 'lag_1',
 'lag_2',
 'lag_3',
 'lag_4',
 'lag_8',
 'lag_12',
 'lag_26',
 'lag_52',
 'roll_mean_4',
 'roll_mean_8',
 'roll_mean_12',
 'roll_mean_26',
 'roll_std_12',
 'zero_rate_12',
 'price_at_origin',
 'price_was_missing_at_origin',
 'segment',
 'target_month',
 'target_week_sin',
 'target_week_cos',
 'target_quarter']

## 3. Select the objective on internal validation only

Tweedie is the common practitioner default for zero-inflated demand data, so it's tried
alongside L1 and L2 -- but the comparison is scored against `val_mat` (purged internal
validation), **not** the true holdout. The true holdout is not touched anywhere in this
section. `random_state` is fixed for reproducibility.

In [4]:
# Cell - fit all three objectives, compare on the internal validation set only
RANDOM_STATE = 42
LGB_PARAMS = dict(
    n_estimators=2000, learning_rate=0.05, num_leaves=63,
    min_child_samples=50, verbosity=-1, random_state=RANDOM_STATE,
)

obj_results, fitted_models = {}, {}
for objective in ["regression", "regression_l1", "tweedie"]:
    m = lgb.LGBMRegressor(objective=objective, **LGB_PARAMS)
    m.fit(
        train_mat[feature_cols], train_mat["target"],
        eval_X=val_mat[feature_cols], eval_y=val_mat["target"],
        eval_metric="l1", categorical_feature=["segment"],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    )
    val_preds = np.clip(m.predict(val_mat[feature_cols]), 0, None)
    val_score = vn1_score(val_mat["target"], val_preds)
    obj_results[objective] = val_score
    fitted_models[objective] = m
    print(f"{objective:15s} val_vn1_score={val_score:.4f}  best_iter={m.best_iteration_}")

winner = min(obj_results, key=obj_results.get)
print(f"\nwinning objective (internal validation only): {winner}")
model = fitted_models[winner]

regression      val_vn1_score=0.5052  best_iter=93


regression_l1   val_vn1_score=0.6430  best_iter=167


tweedie         val_vn1_score=0.8253  best_iter=54

winning objective (internal validation only): regression


## 4. One honest look at the real 13-week holdout

**Caveat on "honest":** this holdout was already inspected during the first (flawed)
pass at this session, which compared objectives directly against it. This evaluation is
methodologically clean going forward (selection happened on internal validation only,
above), but the holdout itself is not a completely untouched estimate anymore -- it's a
development-holdout result, not a pristine one. Recorded here so it's not lost; the
practical lesson (never touch the true holdout for selection, even once) carries forward
to every later session.

In [5]:
# Cell - predict, clip to non-negative, align, score with the component breakdown
preds = np.clip(model.predict(eval_mat[feature_cols]), 0, None)

forecast = eval_mat[list(KEY) + ["target_date"]].copy()
forecast["forecast"] = preds
forecast = forecast.rename(columns={"target_date": "date"})

scored = score_forecast(test, forecast)
breakdown = vn1_score_breakdown(scored["sales"], scored["forecast"])
for k, v in breakdown.items():
    print(f"{k:15s}: {v:,.4f}" if k != "score" else f"{k:15s}: {v:.4f}")

print("\nfull params:", model.get_params())
print("best_iteration_:", model.best_iteration_)

total_actual   : 3,508,827.0000
total_predicted: 3,423,361.4287
signed_bias    : 85,465.5713
abs_error_sum  : 1,665,184.1507
bias_term      : 85,465.5713
score          : 0.4989

full params: {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0, 'importance_type': 'split', 'learning_rate': 0.05, 'max_depth': -1, 'min_child_samples': 50, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 2000, 'n_jobs': None, 'num_leaves': 63, 'objective': 'regression', 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'verbosity': -1}
best_iteration_: 93


## 5. Compare against the full baseline suite

In [6]:
# Cell - recompute the Session 3 baseline suite on the same holdout for a clean
# side-by-side comparison, alongside the LightGBM score
results = {"LightGBM": breakdown["score"]}

scored_zero = score_forecast(test, zero_forecast(train))
results["zero"] = vn1_score(scored_zero["sales"], scored_zero["forecast"])

for name, window in [("naive-last", 1), ("MA4", 4), ("MA8", 8), ("MA12", 12), ("MA26", 26)]:
    sb = score_forecast(test, ma12_forecast(train, window=window))
    results[name] = vn1_score(sb["sales"], sb["forecast"])

sn = score_forecast(test, seasonal_naive_forecast(df, test, lag_weeks=52))
results["seasonal-naive-52"] = vn1_score(sn["sales"], sn["forecast"])

pd.Series(results, name="vn1_score").sort_values()

MA4                  0.498708
LightGBM             0.498927
MA8                  0.511414
MA12                 0.516309
naive-last           0.524666
MA26                 0.561836
seasonal-naive-52    1.049790
zero                 2.000000
Name: vn1_score, dtype: float64

## 6. Diagnostics: segment contribution, and feature importance

The official aggregate score above stays the primary number - this is diagnostic.
Reporting demand share and error share, not just a bare per-segment score, so a claim
like "segment X drags down the average" is actually supported rather than eyeballed --
the official score is not a simple average of segment scores.

In [7]:
# Cell - segment analysis: demand share, absolute-error share, signed bias, score
seg_map = eval_mat.drop_duplicates(subset=list(KEY))[list(KEY) + ["segment"]]
scored_seg = scored.merge(seg_map, on=list(KEY))

total_actual = scored_seg["sales"].sum()
total_abs_err = (scored_seg["sales"] - scored_seg["forecast"]).abs().sum()

for seg_name, g in scored_seg.groupby("segment", observed=True):
    n = g[list(KEY)].drop_duplicates().shape[0]
    demand_share = g["sales"].sum() / total_actual
    abs_err_share = (g["sales"] - g["forecast"]).abs().sum() / total_abs_err
    signed_bias = (g["sales"] - g["forecast"]).sum()
    seg_score = vn1_score(g["sales"], g["forecast"])
    print(f"{seg_name:13s} n_series={n:5d}  demand_share={demand_share:.3f}  "
          f"abs_err_share={abs_err_share:.3f}  signed_bias={signed_bias:+,.0f}  "
          f"score={seg_score:.4f}")

active        n_series= 2006  demand_share=0.454  abs_err_share=0.349  signed_bias=-61,726  score=0.4031
intermittent  n_series= 6450  demand_share=0.440  abs_err_share=0.435  signed_bias=+57,864  score=0.5065
sparse        n_series= 6597  demand_share=0.106  abs_err_share=0.216  signed_bias=+89,328  score=1.2115


In [8]:
# Cell - top feature importances (LightGBM's split-count based default)
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
importance.head(10)

price_at_origin    775
lag_1              604
target_week_sin    449
target_week_cos    437
target_month       376
lag_12             355
lag_26             353
lag_52             337
lag_2              328
lag_8              318
dtype: int32